In [31]:
# NBoW - Neural Bag of Words
# https://github.com/bentrevett/pytorch-sentiment-analysis/blob/main/1%20-%20Neural%20Bag%20of%20Words.ipynb

In [51]:
import collections

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchtext.data
import torchtext.vocab
import tqdm

In [33]:
seed = 1234

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [34]:
train_data, test_data = datasets.load_dataset("imdb", split=["train", "test"])

In [35]:
train_data, test_data

(Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }))

In [36]:
# Tokenization

In [37]:
tokenizer = torchtext.data.utils.get_tokenizer("basic_english")

In [38]:
tokenizer("Hello world! How are you doing today? I'm doing fantastic! Sternocleidomastoid")

['hello',
 'world',
 '!',
 'how',
 'are',
 'you',
 'doing',
 'today',
 '?',
 'i',
 "'",
 'm',
 'doing',
 'fantastic',
 '!',
 'sternocleidomastoid']

In [39]:
def tokenize_example(example, tokenizer, max_length):
    tokens = tokenizer(example["text"])[:max_length]
    return {"tokens": tokens}

In [40]:
?train_data.map

Signature:
train_data.map(
    function: Optional[Callable] = None,
    with_indices: bool = False,
    with_rank: bool = False,
    input_columns: Union[str, List[str], NoneType] = None,
    batched: bool = False,
    batch_size: Optional[int] = 1000,
    drop_last_batch: bool = False,
    remove_columns: Union[str, List[str], NoneType] = None,
    keep_in_memory: bool = False,
    load_from_cache_file: Optional[bool] = None,
    cache_file_name: Optional[str] = None,
    writer_batch_size: Optional[int] = 1000,
    features: Optional[datasets.features.features.Features] = None,
    disable_nullable: bool = False,
    fn_kwargs: Optional[dict] = None,
    num_proc: Optional[int] = None,
    suffix_template: str = '_{rank:05d}_of_{num_proc:05d}',
    new_fingerprint: Optional[str] = None,
    desc: Optional[str] = None,
) -> 'Dataset'
Docstring:
Apply a function to all the examples in the table (individually or in batches) and update the table.
If your function returns a column that al

In [41]:
max_length = 256

train_data = train_data.map(
    tokenize_example, fn_kwargs={"tokenizer": tokenizer, "max_length": max_length}
)
test_data = test_data.map(
    tokenize_example, fn_kwargs={"tokenizer": tokenizer, "max_length": max_length}
)

In [42]:
train_data

Dataset({
    features: ['text', 'label', 'tokens'],
    num_rows: 25000
})

In [43]:
train_data.features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['neg', 'pos'], id=None),
 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)}

In [44]:
train_data[0]["tokens"][:25]

['i',
 'rented',
 'i',
 'am',
 'curious-yellow',
 'from',
 'my',
 'video',
 'store',
 'because',
 'of',
 'all',
 'the',
 'controversy',
 'that',
 'surrounded',
 'it',
 'when',
 'it',
 'was',
 'first',
 'released',
 'in',
 '1967',
 '.']

In [45]:
# Creating Validation Data

In [46]:
test_size = 0.25

train_valid_data = train_data.train_test_split(test_size=test_size)
train_data = train_valid_data["train"]
valid_data = train_valid_data["test"]

In [47]:
len(train_data), len(valid_data), len(test_data)

(18750, 6250, 25000)

In [48]:
# Creating a Vocabulary

In [52]:
min_freq = 5
special_tokens = ["<unk>", "<pad>"]

vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

In [53]:
len(vocab)

21635

In [54]:
vocab.get_itos()[:10]

['<unk>', '<pad>', 'the', '.', ',', 'a', 'and', 'of', 'to', "'"]

In [55]:
vocab["and"]

6

In [56]:
unk_index = vocab["<unk>"]
pad_index = vocab["<pad>"]

In [57]:
"some_token" in vocab

False

In [58]:
vocab.set_default_index(unk_index)

In [59]:
vocab["some_token"]

0

In [60]:
vocab.lookup_indices(["hello", "world", "some_token", "<pad>"])

[5516, 184, 0, 1]